# 02 · Contrastive alignment of images and captions

*Vision-Language Models (VLMs)*, notebook 2 of 4

In notebook 1 the encoder learned from attribute labels. Real image–text data has no such labels, only images and the text that came with them. Contrastive learning, the idea behind CLIP, turns that pairing itself into the training signal: an image's vector should be closer to its own caption's vector than to the other captions in the batch, and the other way round. In this notebook you build a text encoder and the contrastive losses, train both encoders together, and test retrieval on the four held-out combinations.

## What you will build

- `encode_text`: a bag-of-tokens caption encoder;
- `clip_loss`: the symmetric contrastive (InfoNCE) loss with a temperature, as used by CLIP;
- `positive_mask` and `multi_positive_loss`: the same loss when several captions in a batch are correct for one image;
- `recall_at_k`: the retrieval metric, which you use on held-out image–caption pairs;
- a trained pair of encoders whose image vectors retrieve the right caption among all 16 for combinations they never saw.

## Learning objectives

By the end of this notebook you will be able to:

1. encode captions with a bag-of-tokens encoder and say what it cannot represent;
2. compute a temperature-scaled cosine-similarity matrix between image and caption vectors;
3. implement the symmetric InfoNCE loss and explain what each of its two directions asks for;
4. build a positive mask for repeated captions and a multi-positive loss, and explain the loss floor that diagonal-only targets create;
5. evaluate image-to-text retrieval with recall@k on held-out combinations, against an untrained baseline.

### How to work through this notebook

Run the cells from top to bottom. When you reach a **✍️ Your turn** cell, type the function yourself and then run the check cell beneath it:

- ✅ means your function matches the reference, and from then on the notebook runs **your** code;
- ✍️ means you have not written it yet, and ❌ means it does not match yet. In both cases the notebook quietly uses a hidden reference version, so you can always keep going and come back later.

Each reference solution sits in a collapsed cell just above the function you type. Open it only when you are stuck.

## Dependencies

- **Tier 1:** NumPy and Matplotlib only, so the notebook runs in JupyterLite in your browser, on the course JupyterHub server and in Google Colab.
- Nothing is downloaded: the images and captions come from the generator you met in notebook 1.
- Training uses the tiny tensor engine from Base Model Training, defined in a hidden recap cell.
- Every random number comes from a seeded generator. Training takes well under a second, and the whole notebook runs in a few seconds.

## Setup

Run this cell first. It imports NumPy, `time` and Matplotlib and defines `practice`, the helper that checks each function you write.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import time

import matplotlib.pyplot as plt

## 📦 Recap: the tiny tensor engine you built in Base Model Training

The hidden cell below defines the reverse-mode autograd engine from Base Model Training, unchanged: `Tensor` (a NumPy array that remembers how it was computed, so that `backward()` can fill in `.grad`), `parameter`, `zeros_parameter`, `zero_grad`, `cross_entropy`, `layer_norm`, `numerical_grad` and `relative_error`. Every operation in it is gradient-checked. Run the cell; open it only if you want to look back at the code.

In [ ]:
# ── Tiny tensor autograd (NumPy) ── the engine you built in Base Model Training ──
import numpy as np


def _unbroadcast(grad, shape):
    """Sum a gradient back down to `shape`, undoing NumPy broadcasting."""
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for axis, size in enumerate(shape):
        if size == 1 and grad.shape[axis] != 1:
            grad = grad.sum(axis=axis, keepdims=True)
    return grad


class Tensor:
    """A NumPy array that remembers how it was made, so gradients can flow back."""

    def __init__(self, data, requires_grad=False, parents=(), backward=None, op=""):
        self.data = np.asarray(data, dtype=np.float64)
        self.requires_grad = requires_grad
        self.grad = None
        self._parents = parents
        self._backward = backward  # grad_out -> one gradient per parent
        self.op = op

    shape = property(lambda self: self.data.shape)
    ndim = property(lambda self: self.data.ndim)
    T = property(lambda self: self.transpose())

    def __repr__(self):
        return f"Tensor(shape={self.shape}, op={self.op or 'leaf'}, requires_grad={self.requires_grad})"

    @staticmethod
    def lift(value):
        return value if isinstance(value, Tensor) else Tensor(value)

    def _result(self, data, parents, backward, op):
        if any(parent.requires_grad for parent in parents):
            return Tensor(data, True, parents, backward, op)
        return Tensor(data, op=op)

    # arithmetic, with broadcasting
    def __add__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data + other.data, (self, other), lambda g: (g, g), "add")

    __radd__ = __add__

    def __neg__(self):
        return self._result(-self.data, (self,), lambda g: (-g,), "neg")

    def __sub__(self, other):
        return self + (-Tensor.lift(other))

    def __rsub__(self, other):
        return Tensor.lift(other) + (-self)

    def __mul__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data * other.data, (self, other),
                            lambda g: (g * other.data, g * self.data), "mul")

    __rmul__ = __mul__

    def __truediv__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data / other.data, (self, other),
                            lambda g: (g / other.data, -g * self.data / other.data ** 2), "div")

    def __rtruediv__(self, other):
        return Tensor.lift(other) / self

    def __pow__(self, exponent):
        assert isinstance(exponent, (int, float)), "only constant exponents"
        return self._result(self.data ** exponent, (self,),
                            lambda g: (g * exponent * self.data ** (exponent - 1),), "pow")

    def __matmul__(self, other):
        other = Tensor.lift(other)

        def backward(g):
            return (g @ np.swapaxes(other.data, -1, -2), np.swapaxes(self.data, -1, -2) @ g)

        return self._result(self.data @ other.data, (self, other), backward, "matmul")

    # elementwise functions
    def exp(self):
        out = np.exp(self.data)
        return self._result(out, (self,), lambda g: (g * out,), "exp")

    def log(self):
        return self._result(np.log(self.data), (self,), lambda g: (g / self.data,), "log")

    def tanh(self):
        out = np.tanh(self.data)
        return self._result(out, (self,), lambda g: (g * (1 - out ** 2),), "tanh")

    def relu(self):
        return self._result(np.maximum(self.data, 0), (self,), lambda g: (g * (self.data > 0),), "relu")

    def sigmoid(self):
        out = 0.5 * (1 + np.tanh(0.5 * self.data))  # stable for large |x|
        return self._result(out, (self,), lambda g: (g * out * (1 - out),), "sigmoid")

    # reductions and reshaping
    def sum(self, axis=None, keepdims=False):
        def backward(g):
            if axis is not None and not keepdims:
                g = np.expand_dims(g, axis)
            return (np.broadcast_to(g, self.shape).copy(),)

        return self._result(self.data.sum(axis=axis, keepdims=keepdims), (self,), backward, "sum")

    def mean(self, axis=None, keepdims=False):
        count = self.data.size if axis is None else np.prod([self.shape[a] for a in np.atleast_1d(axis)])
        return self.sum(axis=axis, keepdims=keepdims) / count

    def reshape(self, *shape):
        return self._result(self.data.reshape(*shape), (self,), lambda g: (g.reshape(self.shape),), "reshape")

    def transpose(self, *axes):
        axes = axes or tuple(reversed(range(self.ndim)))
        inverse = np.argsort(axes)
        return self._result(self.data.transpose(axes), (self,), lambda g: (g.transpose(inverse),), "transpose")

    def __getitem__(self, index):
        def backward(g):
            full = np.zeros_like(self.data)
            np.add.at(full, index, g)  # repeated indices (an embedding row used twice) add up
            return (full,)

        return self._result(self.data[index], (self,), backward, "index")

    def log_softmax(self, axis=-1):
        shifted = self.data - self.data.max(axis=axis, keepdims=True)
        out = shifted - np.log(np.exp(shifted).sum(axis=axis, keepdims=True))
        probs = np.exp(out)
        return self._result(out, (self,), lambda g: (g - probs * g.sum(axis=axis, keepdims=True),), "log_softmax")

    def softmax(self, axis=-1):
        shifted = np.exp(self.data - self.data.max(axis=axis, keepdims=True))
        out = shifted / shifted.sum(axis=axis, keepdims=True)
        return self._result(out, (self,), lambda g: (out * (g - (g * out).sum(axis=axis, keepdims=True)),), "softmax")

    # reverse-mode differentiation
    def backward(self, grad=None):
        order, seen = [], set()

        def visit(node):
            if id(node) not in seen:
                seen.add(id(node))
                for parent in node._parents:
                    visit(parent)
                order.append(node)

        visit(self)
        self.grad = np.ones_like(self.data) if grad is None else np.asarray(grad, dtype=np.float64)
        for node in reversed(order):
            if node._backward is None or node.grad is None:
                continue
            for parent, parent_grad in zip(node._parents, node._backward(node.grad)):
                if not parent.requires_grad or parent_grad is None:
                    continue
                parent_grad = _unbroadcast(np.asarray(parent_grad, dtype=np.float64), parent.shape)
                parent.grad = parent_grad if parent.grad is None else parent.grad + parent_grad


def parameter(*shape, rng, scale=None):
    """A trainable weight, initialised with N(0, scale^2); default scale is He's sqrt(2 / fan_in)."""
    scale = np.sqrt(2.0 / shape[0]) if scale is None else scale
    return Tensor(rng.normal(0.0, scale, shape), requires_grad=True)


def zeros_parameter(*shape):
    return Tensor(np.zeros(shape), requires_grad=True)


def zero_grad(params):
    for p in params:
        p.grad = None


def cross_entropy(logits, targets, mask=None):
    """Mean next-token/class cross-entropy. logits: (..., classes); targets: int (...); mask: 1.0 keeps a position."""
    flat = logits.reshape(-1, logits.shape[-1]).log_softmax(axis=-1)
    targets = np.asarray(targets).reshape(-1)
    picked = flat[np.arange(targets.size), targets]
    if mask is None:
        return -picked.mean()
    weights = np.asarray(mask, dtype=np.float64).reshape(-1)
    return -(picked * weights).sum() / max(weights.sum(), 1.0)


def layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(axis=-1, keepdims=True)
    centred = x - mean
    var = (centred ** 2).mean(axis=-1, keepdims=True)
    return centred / (var + eps) ** 0.5 * gamma + beta


def numerical_grad(f, tensor, eps=1e-6):
    """Central-difference gradient of scalar f() with respect to tensor.data."""
    grad = np.zeros_like(tensor.data)
    it = np.nditer(tensor.data, flags=["multi_index"])
    for _ in it:
        index = it.multi_index
        original = tensor.data[index]
        tensor.data[index] = original + eps
        plus = float(f().data)
        tensor.data[index] = original - eps
        minus = float(f().data)
        tensor.data[index] = original
        grad[index] = (plus - minus) / (2 * eps)
    return grad


def relative_error(a, b):
    return float(np.linalg.norm(a - b) / max(np.linalg.norm(a) + np.linalg.norm(b), 1e-12))

## 📦 Recap: what you built earlier

The hidden cell below re-defines the pieces from notebook 1 under their usual names, so this notebook works on its own: `practice_tensor`, the scene generator (`sample_scenes`, `render`, `describe`, `combo_ids` and the attribute constants), the three sets `train_scenes`, `seen_scenes` and `held_out_scenes` with their images, `channel_stats`, `normalise_images`, `patchify`, `unpatchify`, `prepare`, `init_image_encoder`, `embed_patches`, `encode_images` and the `Adam` optimiser. Run it; open it only if you want to look back at the code.

In [ ]:
def _fresh(value):
    """A copy of every Tensor inside value, so each check starts from clean gradients."""
    if isinstance(value, Tensor):
        return Tensor(value.data.copy(), requires_grad=value.requires_grad)
    if isinstance(value, dict):
        return {key: _fresh(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return type(value)(_fresh(item) for item in value)
    return value


def _trainable_leaves(value):
    if isinstance(value, Tensor):
        return [value] if value.requires_grad else []
    if isinstance(value, dict):
        return [leaf for key in sorted(value) for leaf in _trainable_leaves(value[key])]
    if isinstance(value, (list, tuple)):
        return [leaf for item in value for leaf in _trainable_leaves(item)]
    return []


def practice_tensor(name, yours, reference, cases, atol=1e-6):
    """practice() for functions that build Tensors.

    Each case runs on fresh copies of its Tensor arguments. The check compares
    the values your function returns AND the gradients that flow back to every
    argument with requires_grad=True, so a function that cuts the graph (for
    example by working on .data) is caught as well.
    """
    def checked(function):
        def run(*args):
            copies = [_fresh(arg) for arg in args]
            out = function(*copies)
            outs = out if isinstance(out, tuple) else (out,)
            if not all(isinstance(item, Tensor) for item in outs):
                raise TypeError("return Tensor results built from Tensor operations, so gradients can flow")
            total = Tensor(0.0)
            for index, item in enumerate(outs):
                weights = np.random.default_rng(index).normal(size=item.shape)
                total = total + (item * weights).sum()
            total.backward()
            grads = [leaf.grad if leaf.grad is not None else np.zeros_like(leaf.data)
                     for leaf in _trainable_leaves(copies)]
            return {"values": [item.data for item in outs], "gradients": grads}
        return run

    mine, ref = checked(yours), checked(reference)
    return yours if practice(name, mine, ref, cases, atol) is mine else reference


COLOURS = ("red", "green", "blue", "yellow")
SIZES = ("small", "large")
ORIENTATIONS = ("horizontal", "vertical")
ATTRIBUTES = {"colour": COLOURS, "size": SIZES, "orientation": ORIENTATIONS}
RGB = np.array([[0.85, 0.12, 0.10],   # red
                [0.15, 0.70, 0.20],   # green
                [0.20, 0.30, 0.90],   # blue
                [0.90, 0.80, 0.10]])  # yellow
BAR = {0: (2, 8), 1: (3, 12)}  # size index -> (thickness, length) in pixels
IMAGE_SIZE = 16

# Every (colour, size, orientation) combination, as attribute indices.
ALL_COMBOS = [(c, s, o) for c in range(4) for s in range(2) for o in range(2)]
# Four combinations never shown in training. Every single attribute value
# still appears in training; only these particular combinations are new.
HELD_OUT_COMBOS = [(0, 0, 1), (1, 1, 0), (2, 0, 0), (3, 1, 1)]
TRAIN_COMBOS = [combo for combo in ALL_COMBOS if combo not in HELD_OUT_COMBOS]


def bar_box(size, orientation):
    """(height, width) of the bar in pixels."""
    thickness, length = BAR[int(size)]
    return (thickness, length) if orientation == 0 else (length, thickness)


def sample_scenes(n, combos, seed):
    """Describe n images: attribute indices, the bar's top-left corner and a noise seed."""
    rng = np.random.default_rng(seed)
    chosen = np.array(combos)[rng.integers(0, len(combos), n)]
    scenes = {"colour": chosen[:, 0], "size": chosen[:, 1], "orientation": chosen[:, 2]}
    boxes = np.array([bar_box(s, o) for s, o in zip(scenes["size"], scenes["orientation"])])
    scenes["row"] = rng.integers(0, IMAGE_SIZE - boxes[:, 0] + 1)
    scenes["col"] = rng.integers(0, IMAGE_SIZE - boxes[:, 1] + 1)
    scenes["noise_seed"] = rng.integers(0, 2**31, n)
    return scenes


def render(scenes):
    """Draw every scene as a (16, 16, 3) RGB image with values in [0, 1]."""
    n = len(scenes["colour"])
    images = np.empty((n, IMAGE_SIZE, IMAGE_SIZE, 3))
    for i in range(n):
        rng = np.random.default_rng(int(scenes["noise_seed"][i]))
        image = 0.08 + 0.06 * rng.random((IMAGE_SIZE, IMAGE_SIZE, 3))  # dark, slightly noisy background
        shade = rng.uniform(0.85, 1.15)  # brightness varies from image to image
        tint = rng.uniform(-0.1, 0.1, 3)  # and so does the exact hue
        colour = RGB[scenes["colour"][i]] * shade + tint
        height, width = bar_box(scenes["size"][i], scenes["orientation"][i])
        r, c = scenes["row"][i], scenes["col"][i]
        image[r:r + height, c:c + width] = colour + 0.04 * rng.normal(size=(height, width, 3))
        images[i] = np.clip(image, 0.0, 1.0)
    return images


def describe(scenes):
    """The caption of every scene, for example 'a large red horizontal bar'."""
    return [f"a {SIZES[s]} {COLOURS[c]} {ORIENTATIONS[o]} bar"
            for c, s, o in zip(scenes["colour"], scenes["size"], scenes["orientation"])]


def combo_ids(scenes):
    """One number from 0 to 15 per scene: which of the 16 combinations it shows."""
    return scenes["colour"] * 4 + scenes["size"] * 2 + scenes["orientation"]


train_scenes = sample_scenes(800, TRAIN_COMBOS, seed=101)
seen_scenes = sample_scenes(300, TRAIN_COMBOS, seed=202)        # new images, combinations seen in training
held_out_scenes = sample_scenes(300, HELD_OUT_COMBOS, seed=303)  # new images, combinations never seen
train_images, seen_images, held_out_images = render(train_scenes), render(seen_scenes), render(held_out_scenes)


def channel_stats(images):
    """Mean and standard deviation of each colour channel, over every pixel of every image."""
    return images.mean(axis=(0, 1, 2)), images.std(axis=(0, 1, 2))


def normalise_images(images, mean, std):
    """(images - mean) / std for every channel. images: (n, H, W, 3); mean, std: (3,)."""
    return (images - mean) / std


def patchify(images, patch_size):
    """Cut images (n, H, W, C) into non-overlapping square patches: (n, N, P*P*C), N = (H/P)*(W/P).

    Patches are numbered row by row; inside a patch the values run row by row,
    then column by column, then channel by channel.
    """
    n, height, width, channels = images.shape
    p = patch_size
    grid = images.reshape(n, height // p, p, width // p, p, channels)
    grid = grid.transpose(0, 1, 3, 2, 4, 5)  # (n, rows, cols, p, p, C)
    return grid.reshape(n, (height // p) * (width // p), p * p * channels)


def unpatchify(patches, patch_size, height, width):
    """The inverse of patchify: (n, N, P*P*C) back to images (n, H, W, C)."""
    n, _, patch_dim = patches.shape
    p = patch_size
    channels = patch_dim // (p * p)
    grid = patches.reshape(n, height // p, width // p, p, p, channels)
    grid = grid.transpose(0, 1, 3, 2, 4, 5)  # (n, rows, p, cols, p, C)
    return grid.reshape(n, height, width, channels)


def prepare(images, mean, std, patch_size=4):
    """Normalise with training statistics, then cut into patches."""
    return patchify(normalise_images(images, mean, std), patch_size)


def init_image_encoder(seed, patch_dim=48, n_patches=16, width=32):
    """Weights of the tiny image encoder, all trainable Tensors."""
    rng = np.random.default_rng(seed)
    return {
        "W_patch": parameter(patch_dim, width, rng=rng),
        "b_patch": zeros_parameter(width),
        "pos": parameter(n_patches, width, rng=rng, scale=0.1),
        "W_mix": parameter(width, width, rng=rng),
        "b_mix": zeros_parameter(width),
    }


def embed_patches(patches, W, b, pos):
    """Linear patch embedding plus one learned position vector per patch.

    patches: Tensor (n, N, patch_dim); W: (patch_dim, D); b: (D,); pos: (N, D).
    Returns a Tensor (n, N, D).
    """
    return patches @ W + b + pos


def encode_images(patches, params):
    """The tiny encoder: embed the patches, apply a ReLU layer to each patch, then average.

    patches: Tensor (n, N, patch_dim); params: the dictionary from init_image_encoder.
    Returns (features, pooled): Tensors of shape (n, N, D) and (n, D).
    """
    tokens = embed_patches(patches, params["W_patch"], params["b_patch"], params["pos"])
    features = (tokens @ params["W_mix"] + params["b_mix"]).relu()
    return features, features.mean(axis=1)


class Adam:
    """The Adam optimiser for a list of Tensors, as in frameworks such as PyTorch."""

    def __init__(self, params, lr=0.01, betas=(0.9, 0.999), eps=1e-8):
        self.params, self.lr, self.betas, self.eps = list(params), lr, betas, eps
        self.m = [np.zeros_like(p.data) for p in self.params]  # running mean of gradients
        self.v = [np.zeros_like(p.data) for p in self.params]  # running mean of squared gradients
        self.t = 0

    def zero_grad(self):
        zero_grad(self.params)

    def step(self):
        self.t += 1
        beta1, beta2 = self.betas
        for p, m, v in zip(self.params, self.m, self.v):
            if p.grad is None:
                continue
            m[...] = beta1 * m + (1 - beta1) * p.grad
            v[...] = beta2 * v + (1 - beta2) * p.grad ** 2
            m_hat, v_hat = m / (1 - beta1 ** self.t), v / (1 - beta2 ** self.t)
            p.data -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

In [ ]:
mean, std = channel_stats(train_images)
train_patches, seen_patches, held_out_patches = (prepare(images, mean, std)
                                                 for images in (train_images, seen_images, held_out_images))
train_combo_ids = combo_ids(train_scenes)
print("training patches:", train_patches.shape, " first captions:", describe(train_scenes)[:2])

## Part 1 · Turn captions into vectors

A caption is first split into words and each word replaced by its id in a vocabulary. Our vocabulary has 11 entries: `<pad>` (id 0), `a`, `bar`, the two sizes, the four colours and the two orientations. Every caption is padded with `<pad>` to length 6, so a batch of captions is an integer array of shape (n, 6). The tokeniser is given:

In [ ]:
VOCAB = ["<pad>", "a", "bar"] + list(SIZES) + list(COLOURS) + list(ORIENTATIONS)
TOKEN_ID = {word: index for index, word in enumerate(VOCAB)}
CAPTION_LENGTH = 6  # the longest caption has 5 words; the rest is padding


def tokenise(captions, length=CAPTION_LENGTH):
    """Word ids, padded with 0 (<pad>) to `length`: an int array (n, length)."""
    ids = np.zeros((len(captions), length), dtype=int)
    for i, caption in enumerate(captions):
        words = caption.split()
        ids[i, :len(words)] = [TOKEN_ID[word] for word in words]
    return ids


example = tokenise(["a large red horizontal bar", "a small blue vertical bar"])
print(VOCAB)
print(example)

A **bag-of-tokens** encoder looks up a learned vector for each real token, averages them, and maps the average with a matrix. With an embedding table $E$ (one row per vocabulary entry) and the set $R$ of real, non-pad positions in a caption,

$$\mathbf{t} = \Big(\frac{1}{|R|} \sum_{j \in R} E_{w_j}\Big) W.$$

Shapes: ids (n, L) → looked-up vectors (n, L, D) → average (n, D) → output (n, D_out). The padding positions must not count: a caption of five words padded to six must give the same vector as the same five words padded to ten. Real systems use a transformer text encoder instead, which also takes word order into account; a bag of tokens cannot, and you will see that in a moment.

### ✍️ Your turn: `encode_text`

Write `encode_text(token_ids, E, W)`.

- `token_ids`: an integer NumPy array of shape (n, L), with 0 for `<pad>`;
- `E`: a `Tensor` of shape (V, D), one row per vocabulary entry; `W`: a `Tensor` of shape (D, D_out);
- return a `Tensor` of shape (n, D_out): the mean of the real tokens' rows of `E`, multiplied by `W`.

*Hints:* `E[token_ids]` looks up all the rows at once, giving shape (n, L, D). Build a mask `(token_ids != 0)` as floats with shape (n, L, 1), multiply, sum over axis 1 and divide by the number of real tokens in each caption.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def encode_text_reference(token_ids, E, W):
    """Bag-of-tokens text encoder: average the embeddings of the real (non-pad) tokens, then map.

    token_ids: int array (n, L) with 0 for <pad>; E: Tensor (V, D); W: Tensor (D, D_out).
    Returns a Tensor (n, D_out).
    """
    mask = (token_ids != 0).astype(float)[..., None]  # (n, L, 1): 1.0 for real tokens
    mean = (E[token_ids] * mask).sum(axis=1) / mask.sum(axis=1)  # (n, D)
    return mean @ W

In [ ]:
def encode_text(token_ids, E, W):
    """Bag-of-tokens text encoder: average the embeddings of the real (non-pad) tokens, then map.

    token_ids: int array (n, L) with 0 for <pad>; E: Tensor (V, D); W: Tensor (D, D_out).
    Returns a Tensor (n, D_out).
    """
    # ✍️ Your code here (three lines)
    raise NotImplementedError

In [ ]:
case_rng = np.random.default_rng(21)
E_case, W_case = parameter(len(VOCAB), 4, rng=case_rng), parameter(4, 3, rng=case_rng)
encode_text = practice_tensor("encode_text", encode_text, encode_text_reference, cases=[
    (tokenise(["a large red horizontal bar", "a small blue vertical bar"]), E_case, W_case),
    (np.array([[1, 3, 0, 0, 0, 0], [5, 0, 0, 0, 0, 0]]), E_case, W_case),  # two and one real tokens
    (np.array([[2, 2, 2, 2, 2, 2]]), E_case, W_case),  # edge case: no padding at all
])

**Expected behaviour.** ✅ on three cases. If you average over all six positions, the padding drags every vector towards the `<pad>` row and the second case fails. If you forget `@ W`, the output has width 4 instead of 3.

A bag of tokens ignores order. The cell below encodes three captions made of the same five words in different orders, with random weights: all three vectors are identical. That is harmless for our one-object captions, but a bag-of-tokens model could never tell "a red bar left of a blue bar" from "a blue bar left of a red bar".

In [ ]:
demo_rng = np.random.default_rng(22)
E_demo, W_demo = parameter(len(VOCAB), 16, rng=demo_rng), parameter(16, 16, rng=demo_rng)
vectors = encode_text(tokenise(["a large red horizontal bar", "a red large horizontal bar", "bar horizontal red large a"]),
                      E_demo, W_demo).data
print("largest difference between the three vectors:", np.abs(vectors - vectors[0]).max())

## Part 2 · Cosine similarity and the temperature

Both encoders end in vectors of the same width, 16 here. Scaling every vector to length 1 makes the dot product of an image vector $\mathbf{u}_i$ and a caption vector $\mathbf{v}_j$ their **cosine similarity**, a number between −1 and 1. For a batch of $n$ pairs this gives an $n \times n$ matrix, images in rows and captions in columns, with the matching pairs on the diagonal. Dividing by a **temperature** $\tau$ turns similarities into logits:

$$\ell_{ij} = \frac{\mathbf{u}_i \cdot \mathbf{v}_j}{\tau}, \qquad \|\mathbf{u}_i\| = \|\mathbf{v}_j\| = 1.$$

With $\tau = 0.1$ the logits lie between −10 and 10, enough for a softmax to become confident. A large temperature keeps every probability close to uniform; a very small one makes the loss focus on the few hardest pairs. CLIP learns the temperature during training, starting from 0.07; here it is fixed so that you can change it by hand. `l2_normalise` is given:

In [ ]:
def l2_normalise(x, eps=1e-8):
    """Scale every row of x to unit length, so dot products become cosine similarities."""
    return x / ((x * x).sum(axis=-1, keepdims=True) + eps) ** 0.5


demo_rng = np.random.default_rng(23)
image_vectors = l2_normalise(Tensor(demo_rng.normal(size=(3, 16)))).data
caption_vectors = l2_normalise(Tensor(demo_rng.normal(size=(3, 16)))).data
similarity = image_vectors @ caption_vectors.T
print("cosine similarities (images in rows, captions in columns):\n", similarity.round(2))
print("logits at temperature 0.1:\n", (similarity / 0.1).round(1))

## Part 3 · The symmetric contrastive loss (InfoNCE)

Treat each row of the logit matrix as a classification problem: image $i$ must pick its own caption out of the $n$ captions in the batch. Treat each column the same way: caption $j$ must pick its own image. The **symmetric InfoNCE** loss averages the two cross-entropies:

$$\mathcal{L} = \frac{1}{2}\left[\frac{1}{n}\sum_{i=1}^{n} -\log \frac{e^{\ell_{ii}}}{\sum_{j} e^{\ell_{ij}}} \; + \; \frac{1}{n}\sum_{j=1}^{n} -\log \frac{e^{\ell_{jj}}}{\sum_{i} e^{\ell_{ij}}}\right].$$

The first term is image-to-text (softmax along each row), the second text-to-image (softmax down each column). Every other caption in the batch acts as a negative for image $i$, so a larger batch gives each image more competitors. The demo shows the row softmax for a 3 × 3 logit matrix:

In [ ]:
logits_demo = np.array([[5.0, 1.0, 0.0], [0.5, 4.0, 1.0], [0.0, 0.0, 3.0]])
row_probs = np.exp(logits_demo) / np.exp(logits_demo).sum(axis=1, keepdims=True)
print("image-to-text probabilities, each row sums to 1:\n", row_probs.round(3))

### ✍️ Your turn: `clip_loss`

Write `clip_loss(logits)`.

- `logits`: a `Tensor` of shape (n, n), images in rows and captions in columns, with the matching pair of row $i$ in column $i$;
- return the scalar `Tensor` $\mathcal{L}$ above.

*Hints:* the engine's `cross_entropy(logits, targets)` already computes the mean of $-\log$ softmax along the last axis, picking the entry named by each integer target. The targets for both directions are `np.arange(n)`; the text-to-image direction is the same computation on `logits.T`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def clip_loss_reference(logits):
    """Symmetric InfoNCE with the matching pair on the diagonal. logits: Tensor (n, n) -> scalar Tensor."""
    targets = np.arange(logits.shape[0])
    return (cross_entropy(logits, targets) + cross_entropy(logits.T, targets)) * 0.5

In [ ]:
def clip_loss(logits):
    """Symmetric InfoNCE with the matching pair on the diagonal. logits: Tensor (n, n) -> scalar Tensor."""
    # ✍️ Your code here (two lines)
    raise NotImplementedError

In [ ]:
case_rng = np.random.default_rng(24)
clip_loss = practice_tensor("clip_loss", clip_loss, clip_loss_reference, cases=[
    (Tensor(logits_demo, requires_grad=True),),
    (parameter(5, 5, rng=case_rng, scale=2.0),),
    (parameter(1, 1, rng=case_rng),),  # edge case: one pair, so the loss is exactly 0
])
print("loss for the demo logits:", round(float(clip_loss(Tensor(logits_demo)).data), 4))
print("loss for all-zero 64 x 64 logits:", round(float(clip_loss(Tensor(np.zeros((64, 64)))).data), 4),
      " log(64) =", round(float(np.log(64)), 4))

**Expected behaviour.** ✅ on three cases, including a single pair, whose loss is exactly 0. When every logit is equal, each softmax is uniform over 64 candidates and the loss is $\log 64 \approx 4.159$: the value an untrained model starts near. If you only compute one direction, the values differ from the reference whenever the matrix is not symmetric.

## Part 4 · When several captions are correct

Our training set has only 12 distinct captions, so a batch of 64 pairs contains each caption about five times. Diagonal targets then say that image $i$ must choose caption $i$ and **not** caption $k$, even when caption $k$ is the very same sentence. Web data has a milder version of this problem: duplicated alt text and near-identical images. The cell measures the effect on a "perfect" model whose image and caption vectors are identical one-hot vectors for each combination. Such a model cannot do better, yet the diagonal-only loss stays far above zero.

In [ ]:
batch = np.random.default_rng(25).choice(len(train_combo_ids), 64, replace=False)
batch_ids = train_combo_ids[batch]
values, counts = np.unique(batch_ids, return_counts=True)
print(f"64 pairs, {len(values)} distinct captions; copies of each: {sorted(counts.tolist(), reverse=True)}")
perfect = Tensor(np.eye(16)[batch_ids])  # one-hot vectors already have length 1
perfect_logits = perfect @ perfect.T / 0.1
copies = counts[np.searchsorted(values, batch_ids)]
print("diagonal-only loss of the perfect model:", round(float(clip_loss(perfect_logits).data), 3))
print("average log(copies of the caption):     ", round(float(np.log(copies).mean()), 3))

**What to notice.** The perfect model's diagonal-only loss is about the average of $\log(\text{copies})$: with $k$ identical correct captions, the best the softmax can do is split its probability equally between them, so $-\log(1/k) = \log k$ is a floor that no training can remove. The fix is to say which pairs are correct. The **positive mask** has $M_{ij} = 1$ when items $i$ and $j$ carry the same caption, and the **multi-positive loss** asks for the total probability on all correct matches:

$$\mathcal{L}_{\text{multi}} = \frac{1}{2}\left[\frac{1}{n}\sum_{i} -\log \sum_{j:\,M_{ij}=1} p^{\,\text{row}}_{ij} \; + \; \frac{1}{n}\sum_{j} -\log \sum_{i:\,M_{ij}=1} p^{\,\text{col}}_{ij}\right],$$

where $p^{\,\text{row}}$ is the softmax along each row and $p^{\,\text{col}}$ the softmax down each column. With $M$ equal to the identity matrix it is exactly `clip_loss`. Supervised contrastive learning (Khosla et al., 2020) studies closely related ways of using several positives.

### ✍️ Your turn: `positive_mask`

Write `positive_mask(caption_ids)`.

- `caption_ids`: an integer array of shape (n,), one caption (or combination) id per pair;
- return a float array of shape (n, n) with 1.0 where `caption_ids[i] == caption_ids[j]` and 0.0 elsewhere. The diagonal is always 1.

*Hint:* compare a column view `ids[:, None]` with a row view `ids[None, :]`; broadcasting builds the whole table in one step.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def positive_mask_reference(caption_ids):
    """mask[i, j] = 1.0 when items i and j have the same caption, else 0.0. caption_ids: (n,) -> (n, n)."""
    ids = np.asarray(caption_ids)
    return (ids[:, None] == ids[None, :]).astype(float)

In [ ]:
def positive_mask(caption_ids):
    """mask[i, j] = 1.0 when items i and j have the same caption, else 0.0. caption_ids: (n,) -> (n, n)."""
    # ✍️ Your code here (one or two lines)
    raise NotImplementedError

In [ ]:
positive_mask = practice("positive_mask", positive_mask, positive_mask_reference, cases=[
    (np.array([0, 1, 0]),),
    (np.array([7, 7, 7, 2]),),
    (np.array([5]),),  # edge case: a single pair
])
print(positive_mask(np.array([0, 1, 0])))

### ✍️ Your turn: `multi_positive_loss`

Write `multi_positive_loss(logits, positives)`.

- `logits`: a `Tensor` of shape (n, n), images in rows and captions in columns;
- `positives`: a float array of shape (n, n) from `positive_mask`;
- return the scalar `Tensor` $\mathcal{L}_{\text{multi}}$ above.

*Hints:* `logits.softmax(axis=1)` gives the row probabilities. Multiply by `positives`, sum along the row, then take `.log()` and the mean; repeat with `axis=0` for the columns. Here the logits are cosine similarities divided by $\tau$, so they stay between $-1/\tau$ and $1/\tau$ and taking the log of a probability is safe; production code works with log-probabilities throughout instead.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def multi_positive_loss_reference(logits, positives):
    """Symmetric contrastive loss in which every 1.0 in `positives` counts as a correct match.

    logits: Tensor (n, n), images in rows and captions in columns; positives: (n, n) array.
    """
    image_to_text = -(logits.softmax(axis=1) * positives).sum(axis=1).log().mean()
    text_to_image = -(logits.softmax(axis=0) * positives).sum(axis=0).log().mean()
    return (image_to_text + text_to_image) * 0.5

In [ ]:
def multi_positive_loss(logits, positives):
    """Symmetric contrastive loss in which every 1.0 in `positives` counts as a correct match.

    logits: Tensor (n, n), images in rows and captions in columns; positives: (n, n) array.
    """
    # ✍️ Your code here (three lines)
    raise NotImplementedError

In [ ]:
case_rng = np.random.default_rng(26)
multi_positive_loss = practice_tensor("multi_positive_loss", multi_positive_loss, multi_positive_loss_reference, cases=[
    (parameter(4, 4, rng=case_rng, scale=2.0), positive_mask(np.array([0, 1, 0, 2]))),
    (parameter(5, 5, rng=case_rng, scale=2.0), np.eye(5)),  # identity mask: the same value as clip_loss
    (parameter(3, 3, rng=case_rng), np.ones((3, 3))),  # edge case: everything matches, so the loss is 0
])
print("multi-positive loss of the perfect model:", round(float(multi_positive_loss(perfect_logits, positive_mask(batch_ids)).data), 5))

**Expected behaviour.** ✅ on three cases. With the identity mask your function equals `clip_loss`, and with an all-ones mask it is 0, because every candidate is correct. The perfect model's multi-positive loss is about 0.0005 instead of the diagonal-only value above 1: the floor has gone, and the loss now measures only real mistakes.

## Part 5 · Measure retrieval with recall@k

Image-to-text retrieval ranks a pool of candidate captions for every image. **Recall@k** is the fraction of queries for which at least one relevant candidate appears among the $k$ best-scored ones. Two choices define what the number means, so always report them:

- **the candidate pool:** here, all 16 possible captions, including the 4 that never appeared in training. Chance recall@1 is therefore $1/16 \approx 0.06$;
- **relevance:** a caption is relevant when it describes the image's combination. With paraphrases or several valid descriptions, more than one candidate can be relevant.

In the demo, the first query's best-scored candidate is index 0, but its relevant candidate is index 2, ranked second. So that query counts for recall@2 but not for recall@1.

In [ ]:
scores_demo = np.array([[0.9, 0.2, 0.7], [0.1, 0.3, 0.8]])
relevant_demo = np.array([[False, False, True], [False, True, False]])
print("candidates from best to worst, per query:\n", np.argsort(-scores_demo, axis=1))

### ✍️ Your turn: `recall_at_k`

Write `recall_at_k(scores, relevant, k)`.

- `scores`: shape (n_queries, n_candidates), higher is better; `relevant`: a boolean array of the same shape;
- return a Python `float`: the fraction of rows with at least one `True` among their `k` highest scores.

*Hints:* `np.argsort(-scores, axis=1, kind="stable")[:, :k]` gives the indices of the top $k$ candidates in each row (on a tie, the lower index ranks first). `np.take_along_axis` reads `relevant` at those indices, and `.any(axis=1)` asks whether each row found a match.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def recall_at_k_reference(scores, relevant, k):
    """Fraction of queries (rows) with at least one relevant candidate among their k best scores.

    scores: (n_queries, n_candidates); relevant: same shape, True where a candidate is a valid match.
    """
    top_k = np.argsort(-scores, axis=1, kind="stable")[:, :k]
    return float(np.take_along_axis(relevant, top_k, axis=1).any(axis=1).mean())

In [ ]:
def recall_at_k(scores, relevant, k):
    """Fraction of queries (rows) with at least one relevant candidate among their k best scores.

    scores: (n_queries, n_candidates); relevant: same shape, True where a candidate is a valid match.
    """
    # ✍️ Your code here (two or three lines)
    raise NotImplementedError

In [ ]:
case_rng = np.random.default_rng(27)
random_scores = case_rng.normal(size=(6, 5))
random_relevant = case_rng.random((6, 5)) < 0.3
recall_at_k = practice("recall_at_k", recall_at_k, recall_at_k_reference, cases=[
    (scores_demo, relevant_demo, 1),
    (scores_demo, relevant_demo, 2),
    (random_scores, random_relevant, 3),
    (random_scores, random_relevant, 5),  # edge case: k equals the number of candidates
])

**Expected behaviour.** ✅ on four cases. For the demo, recall@1 is 0.0 and recall@2 is 1.0. When $k$ equals the pool size, recall@k is simply the fraction of queries that have any relevant candidate at all.

## Put it together: train both encoders

The alignment model joins everything: the image encoder from notebook 1 with a linear head from its 32-number pooled vector to a 16-number embedding, and your bag-of-tokens text encoder. Both outputs are scaled to length 1. Each training step samples 64 image–caption pairs, builds the 64 × 64 logit matrix and takes one Adam step on the multi-positive loss. `retrieval_report` ranks all 16 captions for each test image and returns recall@1, recall@3, and how often the best caption has the right colour, size and orientation separately. Read the given code, then run the cells below it.

In [ ]:
ALL_CAPTIONS = [f"a {SIZES[s]} {COLOURS[c]} {ORIENTATIONS[o]} bar" for c, s, o in ALL_COMBOS]


def init_alignment_model(seed, embed_dim=16):
    """An image encoder with a projection head, and a bag-of-tokens text encoder."""
    rng = np.random.default_rng(seed + 1)
    return {
        "image": init_image_encoder(seed),
        "W_image": parameter(32, embed_dim, rng=rng),
        "E": parameter(len(VOCAB), embed_dim, rng=rng, scale=0.5),
        "W_text": parameter(embed_dim, embed_dim, rng=rng),
    }


def model_parameters(model):
    """Every Tensor in a (nested) dictionary of parameters."""
    found = []
    for value in model.values():
        if isinstance(value, dict):
            found += model_parameters(value)
        elif isinstance(value, Tensor):
            found.append(value)
    return found


def embed_images(model, patches):
    _, pooled = encode_images(Tensor(patches), model["image"])
    return l2_normalise(pooled @ model["W_image"])


def embed_captions(model, token_ids):
    return l2_normalise(encode_text(token_ids, model["E"], model["W_text"]))


def train_alignment(model, patches, token_ids, caption_ids, steps=200, batch_size=64, lr=0.01,
                    temperature=0.1, multi_positive=True, seed=0):
    """Train both encoders with the contrastive loss; returns the loss of every step."""
    optimiser = Adam(model_parameters(model), lr)
    order = np.random.default_rng(seed)
    losses = []
    for step in range(steps):
        batch = order.choice(len(token_ids), batch_size, replace=False)
        logits = embed_images(model, patches[batch]) @ embed_captions(model, token_ids[batch]).T / temperature
        if multi_positive:
            loss = multi_positive_loss(logits, positive_mask(caption_ids[batch]))
        else:
            loss = clip_loss(logits)
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
        losses.append(float(loss.data))
    return losses


def retrieval_report(model, patches, scenes):
    """Image-to-text retrieval over the 16 possible captions: R@1, R@3 and per-attribute top-1 accuracy."""
    scores = embed_images(model, patches).data @ embed_captions(model, tokenise(ALL_CAPTIONS)).data.T
    relevant = combo_ids(scenes)[:, None] == np.arange(len(ALL_CAPTIONS))[None, :]
    best = np.array(ALL_COMBOS)[scores.argmax(axis=1)]
    report = {"R@1": recall_at_k(scores, relevant, 1), "R@3": recall_at_k(scores, relevant, 3)}
    for index, attribute in enumerate(ATTRIBUTES):
        report[attribute] = float((best[:, index] == scenes[attribute]).mean())
    return report

In [ ]:
train_ids = tokenise(describe(train_scenes))
untrained = init_alignment_model(seed=0)
baseline = retrieval_report(untrained, held_out_patches, held_out_scenes)
print("untrained model, held-out images:", {key: round(value, 3) for key, value in baseline.items()})

### 🧪 Try it: temperature and positives

The next cell trains a fresh model. Change `TEMPERATURE` or `MULTI_POSITIVE`, run it and the results cell again, and compare. The checks in the results cell only run with the default settings.

In [ ]:
TEMPERATURE = 0.1      # try 0.02 and 1.0, then run this cell and the next one again
MULTI_POSITIVE = True  # try False

model = init_alignment_model(seed=0)
start = time.perf_counter()
losses = train_alignment(model, train_patches, train_ids, train_combo_ids,
                         temperature=TEMPERATURE, multi_positive=MULTI_POSITIVE, seed=2)
print(f"200 steps in {time.perf_counter() - start:.2f} s; loss {np.mean(losses[:10]):.3f} -> {np.mean(losses[-10:]):.3f}")

In [ ]:
seen_report = retrieval_report(model, seen_patches, seen_scenes)
held_out_report = retrieval_report(model, held_out_patches, held_out_scenes)
print(f"{'image-to-text retrieval':26s}" + "".join(f"{key:>13s}" for key in baseline))
for name, report in (("untrained, held-out", baseline), ("trained, seen", seen_report), ("trained, held-out", held_out_report)):
    print(f"{name:26s}" + "".join(f"{report[key]:13.3f}" for key in baseline))

plt.figure(figsize=(5, 2.6))
plt.plot(losses)
plt.xlabel("step")
plt.ylabel("contrastive loss")
plt.title(f"temperature {TEMPERATURE}, multi-positive {MULTI_POSITIVE}")
plt.show()

if TEMPERATURE == 0.1 and MULTI_POSITIVE:
    assert np.mean(losses[-10:]) < 0.2 * np.mean(losses[:10]), "the loss should fall by at least 80%"
    assert seen_report["R@1"] >= 0.9, "seen combinations should be retrieved almost perfectly"
    assert held_out_report["R@1"] >= 0.75, "held-out recall@1 should be far above chance (1/16)"
    assert held_out_report["R@1"] - baseline["R@1"] >= 0.5, "training should beat the untrained model by a wide margin"
    print("All checks passed.")
else:
    print("Checks skipped: they assume TEMPERATURE = 0.1 and MULTI_POSITIVE = True.")

**Expected behaviour.** Training takes a fraction of a second, and the loss falls from about 1.7 to about 0.01. The untrained model's recall@1 on held-out images is near chance. After training, recall@1 is 1.0 on seen combinations and about 0.96 on held-out ones, and recall@3 is 1.0. The per-attribute columns show that each attribute of the best caption is right at least 98% of the time on held-out images. Across eight seeds, held-out recall@1 ranged from 0.90 to 0.98, so the asserted floor of 0.75 leaves a wide margin.

These captions come from a fixed template, so the text encoder only has to learn 11 word vectors. Real contrastive training uses hundreds of millions of noisy pairs, large batches and transformer encoders, and its scores depend on the candidate pool and on how relevance is defined in the benchmark.

**Temperature.** With `TEMPERATURE = 1.0` the logits stay between −1 and 1, the softmax can never become confident, and the loss stops near 1.56. Across eight seeds, held-out recall@1 was lower than with 0.1 in seven of them, anywhere from 0.33 to 0.98, and almost all of the extra errors were orientation. With `0.02` the loss falls to about 0.003, yet held-out recall@1 was lower than with 0.1 in all eight seeds, between 0.83 and 0.95. A plausible reason is that a very sharp softmax lets a few pairs in each batch dominate the gradient.

## 🧪 Experiments

### 🧪 Experiment 1: diagonal-only targets versus multiple positives

The cell trains two fresh models with the same seed and temperature: one with your multi-positive loss and one with diagonal-only `clip_loss`. It compares their final training losses and their held-out retrieval.

In [ ]:
comparison = {}
for label, multi in (("multi-positive", True), ("diagonal only", False)):
    trial = init_alignment_model(seed=0)
    trial_losses = train_alignment(trial, train_patches, train_ids, train_combo_ids, multi_positive=multi, seed=2)
    comparison[label] = (float(np.mean(trial_losses[-10:])), retrieval_report(trial, held_out_patches, held_out_scenes))
print(f"{'targets':16s} {'final loss':>11s} {'held-out R@1':>13s} {'orientation':>12s}")
for label, (final_loss, report) in comparison.items():
    print(f"{label:16s} {final_loss:11.3f} {report['R@1']:13.3f} {report['orientation']:12.3f}")
assert comparison["diagonal only"][0] > 1.0, "with repeated captions the diagonal-only loss cannot approach 0"
assert comparison["multi-positive"][0] < 0.2

**Expected behaviour.** The diagonal-only loss stays near 1.8, the $\log(\text{copies})$ floor you computed in Part 4, while the multi-positive loss reaches about 0.01. Retrieval, however, comes out about the same for both in this setup: our repeated captions are exact copies with identical text vectors, so the image-to-text direction receives the same push either way. The difference is what the loss *means*. A diagonal-only loss of 1.8 hides whether the model still makes real mistakes, and with paraphrases, which have different text vectors, diagonal targets would actively push correct descriptions apart. Exercise 2 lets you test that.

### 🧪 Experiment 2: look at the mistakes

Aggregate scores hide which images fail. The cell shows held-out images whose best-ranked caption is wrong, with the caption they should have retrieved.

In [ ]:
caption_scores = embed_images(model, held_out_patches).data @ embed_captions(model, tokenise(ALL_CAPTIONS)).data.T
best = caption_scores.argmax(axis=1)
wrong = np.flatnonzero(best != combo_ids(held_out_scenes))
print(f"{len(wrong)} of {len(best)} held-out images retrieve a wrong caption first")
if len(wrong):
    shown = wrong[:6]
    fig, axes = plt.subplots(1, len(shown), figsize=(2.1 * len(shown), 2.6), squeeze=False)
    for ax, i in zip(axes[0], shown):
        ax.imshow(held_out_images[i], interpolation="nearest")
        ax.set_title("got: " + " ".join(ALL_CAPTIONS[best[i]].split()[1:4])
                     + "\ntrue: " + " ".join(describe(held_out_scenes)[i].split()[1:4]), fontsize=7)
        ax.axis("off")
    plt.show()

**Expected behaviour.** Eleven of the 300 held-out images retrieve a wrong caption first, and each of them gets exactly one attribute wrong: most often the orientation, sometimes the colour (a small blue bar read as green) or the size. Look at the pictures: several of these bars touch the image edge or sit across patch boundaries, which is hard for an encoder that never compares neighbouring patches.

## Exercises

1. **✍️ `mean_reciprocal_rank`.** Recall@k ignores *where* in the top $k$ the first match falls. The mean reciprocal rank (MRR) scores a query 1 if its first relevant candidate ranks first, 1/2 if it ranks second, and so on, then averages over queries. Write it below and compute it for the trained model.
2. **Paraphrases.** Add the words `big` and `little` to the vocabulary and caption a third of the training images with them instead of `large` and `small`. Train once with positives defined by the exact caption string and once by the combination. Which one learns that `big` and `large` mean the same thing, and how does held-out retrieval change?
3. **Text-to-image retrieval.** For each of the 16 captions, rank the 300 held-out images and compute the fraction of the top 10 that match (precision@10). Which captions are hardest, and why are only 4 of the 16 captions meaningful queries for this pool?
4. **A learned temperature.** Make $\log(1/\tau)$ a trainable `Tensor` and include it in the optimiser. Where does it settle, and does held-out recall change?

### ✍️ Exercise 1: `mean_reciprocal_rank`

Write `mean_reciprocal_rank(scores, relevant)`: for each row, find the rank (1 = best score) of the first relevant candidate, take $1/\text{rank}$, and return the mean over rows as a Python `float`. You may assume every row has at least one relevant candidate.

*Hint:* sort each row as in `recall_at_k`, read `relevant` in that order, and `argmax` finds the first `True`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def mean_reciprocal_rank_reference(scores, relevant):
    """Mean over queries of 1 / rank of the first relevant candidate (the best score has rank 1).

    scores, relevant: (n_queries, n_candidates); every query has at least one relevant candidate.
    """
    order = np.argsort(-scores, axis=1, kind="stable")
    first_hit = np.take_along_axis(relevant, order, axis=1).argmax(axis=1)  # 0-based position
    return float(np.mean(1.0 / (first_hit + 1)))

In [ ]:
def mean_reciprocal_rank(scores, relevant):
    """Mean over queries of 1 / rank of the first relevant candidate (the best score has rank 1).

    scores, relevant: (n_queries, n_candidates); every query has at least one relevant candidate.
    """
    # ✍️ Your code here (three lines)
    raise NotImplementedError

In [ ]:
with_match = random_relevant.copy()
with_match[:, 0] = True  # every query needs at least one relevant candidate
mean_reciprocal_rank = practice("mean_reciprocal_rank", mean_reciprocal_rank, mean_reciprocal_rank_reference, cases=[
    (scores_demo, relevant_demo),
    (random_scores, with_match),
    (np.array([[0.3, 0.9]]), np.array([[True, True]])),  # edge case: the best candidate is relevant
])
held_out_relevant = combo_ids(held_out_scenes)[:, None] == np.arange(len(ALL_CAPTIONS))[None, :]
print("trained model, held-out MRR:", round(mean_reciprocal_rank(caption_scores, held_out_relevant), 3))

**Expected behaviour.** ✅, with an MRR of 0.5 for the demo (both first matches rank second) and a held-out MRR close to 1 for the trained model, slightly above its recall@1 because most misses rank the right caption second.

## Common errors and debugging

- **Forgetting to normalise.** Without `l2_normalise`, a model can lower the loss by making vectors longer rather than better aligned, and the logits are no longer bounded by $1/\tau$.
- **Rows and columns swapped.** `cross_entropy(logits, targets)` takes the softmax along the last axis, so it treats rows as queries. The text-to-image direction needs `logits.T`.
- **Averaging over padding.** A text encoder that includes `<pad>` positions gives captions of different lengths systematically different vectors.
- **A loss that will not fall below 1.** With repeated captions and diagonal targets, that is the $\log(\text{copies})$ floor, not a bug in the optimiser. Check how many times each caption occurs in a batch.
- **Retrieval scored on training pairs.** Recall on the training images mostly measures memorisation. Report held-out images, state the candidate pool and the relevance rule, and compare with an untrained baseline.
- **`log` of zero.** If a positive's probability underflows to 0, the loss becomes infinite. Keep the temperature moderate, or compute in log space.

## Summary

- A bag-of-tokens encoder averages word vectors; it handles our one-object captions but cannot represent word order.
- Normalised vectors make dot products cosine similarities; the temperature controls how sharp the softmax over candidates becomes.
- The symmetric InfoNCE loss is two cross-entropies, image-to-text along rows and text-to-image down columns, with the matching pairs on the diagonal.
- Repeated captions make diagonal targets wrong; a positive mask and the multi-positive loss define correct matches by meaning and remove the $\log(\text{copies})$ floor.
- Retrieval results depend on the candidate pool and the relevance definition. Here held-out recall@1 was about 0.95 against a chance level of 1/16.

## Next

In **notebook 3** you connect this aligned image encoder to a small language model with a projector, and train it to answer questions such as "what colour is the bar?", with the image encoder frozen and unfrozen.